# TP n°8 — Réseaux de Petri

## Objectifs
Modéliser et vérifier des systèmes concurrents à l’aide des Réseaux de Petri (RdP).

## Rappel
Les RdP constituent un formalisme pour la description et l’analyse des systèmes concurrents.  
L’ensemble minimal de couvertures est une représentation finie de l’ensemble des marquages accessibles. Il peut être utilisé pour résoudre plusieurs problèmes, parmi lesquels déterminer si le réseau est borné (i.e. possède un nombre fini de marquages accessibles).

### Algorithme de Karp et Miller
L’algorithme de Karp et Miller construit un arbre étiqueté par les marquages accessibles :

1. **Initialiser** un arbre avec un sommet étiqueté par le marquage initial.
2. **Tant qu’il existe des feuilles non traitées** dans l’arbre :
   1. Prendre une feuille *f* de l’arbre, étiquetée par *M*.
   2. Si *f* a un ancêtre étiqueté par *M*, passer à la feuille suivante.
   3. Si *f* a un ancêtre étiqueté par *N* tel que  
      ∀q ∈ P, M(q) ≤ N(q) **et** ∃p ∈ P, M(p) < N(p) alors **arrêter** (le réseau est non borné).
   4. Sinon, pour chaque transition *t* admissible par *M* :
      1. Calculer *M′* tel que M [t > M′.
      2. Ajouter un fils à *f* étiqueté par *M′*.
3. Si **toutes les feuilles ont été parcourues**, alors le réseau est **borné**.


## Exercice 1 — Algorithme de Karp et Miller
Vous représentez un RdP par :
- un nombre de places **P** ;
- un ensemble de transitions **T** ;
- un tableau `pre[p][t]` indiquant combien de jetons doivent être retirés de la place *p* lors du franchissement de la transition *t* ;
- un tableau `post[p][t]` indiquant combien de jetons doivent être ajoutés dans la place *p* lors du franchissement de la transition *t*.

Comme un marquage peut apparaître dans plusieurs nœuds de l’arbre, vous devez associer un **identifiant** à chaque nœud. Représentez l’arbre comme :
- un tableau qui, à chaque identifiant, associe un **marquage** ;
- un **dictionnaire** qui, à chaque identifiant, associe son **parent** dans l’arbre. Le marquage initial est son **propre parent**.

**Question 1.** Implémentez l’algorithme de Karp et Miller.

In [17]:
# Rappels : on représente un réseau de Petri par 
## Un nombre de places P
## Un nombre de transitions T
## Un tableau pre[p][t] qui indique combien de jetons doivent être pris dans p quand on emprunte t
## Un tableau post[p][t] qui indique combien de jetons doivent être ajoutés dans p quand on emprunte t

# Un marquage est représenté par un tableau m (m[p]=nombre de jetons dans p)

def apply(P, T, pre, post, m, t):
    #renvoie le marquage obtenu en appliquant, si possible, la transition t au marquage m
    for p in range(P):
        if m[p] < pre[p][t]:
            return None  # transition non franchissable
    return [m[p] - pre[p][t] + post[p][t] for p in range(P)]

def franchissable(P, T, pre, post, m):
    # renvoie la liste des transitions franchissables à partir du marquage m
    return [t for t in range(T) if all(m[p] >= pre[p][t] for p in range(P))]

# L'algorithme de Karp et Miller construit un arbre étiqueté par les marquages accessibles. 
# Comme un marquage peut apparaître dans plusieurs noeuds de l'arbre, on va associer un identifiant à chaque noeud.
# On représente l'arbre comme :
# - un dictionnaire qui à chaque identifiant associe son parent dans l'arbre 
# - un tableau qui à chaque identifiant associe un marquage
# (l'identifiant du marquage initial sera son propre parent)

def ajout_fils(arbre_dict, arbre_tab, iden, m):
    # fonction qui ajoute m dans l'arbre, en tant que fils du noeud iden
    new_id = len(arbre_tab)
    arbre_tab.append(m)
    arbre_dict[new_id] = iden
    return (arbre_dict, arbre_tab)

def ancetres(arbre_dict, arbre_tab, iden):
    # renvoie la liste des ancêtres d'un noeud dans l'arbre
    if iden == 0: 
        return [arbre_tab[iden]]
    else: 
        return ancetres(arbre_dict, arbre_tab, arbre_dict[iden]) + [arbre_tab[iden]]

def temoin(m1, m2):
    # teste si m1 ->* m2 est un témoin de non-bornitude :
    # pour tout p, m1[p] <= m2[p]
    # il existe q, m1[q] < m2[q]
    # (donc renvoie True si "m1 est inclus dans m2", false sinon)
    return (all(m1[p] <= m2[p] for p in range(len(m1))) and
            any(m1[p] < m2[p] for p in range(len(m1))))

def estborne(P, T, pre, post, m):
    #Algo de Karp et Miller

    #Initialiser l'arbre
    arbre_dict = {0: 0}  # le marquage initial est son propre parent
    arbre_tab = [m]
    To_do = [0] 

    while To_do != []:
        iden = To_do.pop(0)
        M = arbre_tab[iden]


        if iden == 0:
            ancs = []  # la racine n'a pas d'ancêtres
        else:
            ancs = ancetres(arbre_dict, arbre_tab, arbre_dict[iden])

        if M in ancs:
            continue

        for N in ancs:
            if temoin(N, M):
                return False


        for t in franchissable(P, T, pre, post, M):
            M_prime = apply(P, T, pre, post, M, t)
            arbre_dict, arbre_tab = ajout_fils(arbre_dict, arbre_tab, iden, M_prime)
            new_id = len(arbre_tab) - 1
            To_do.append(new_id)

    return True

P3 = 2
T3 = 3
pre3 = [[1,0,0],[0,1,1]]
post3 = [[0,1,0],[1,0,0]]
m3 = [1,0]

print(estborne(P3,T3,pre3,post3,m3))

True


## Exercice 2 — Génération du graphe des marquages accessibles
L’algorithme suivant génère le graphe des marquages accessibles \((V, E)\) d’un RdP **borné** :

1. Initialiser une **pile S** avec le marquage initial.
2. **V** contient initialement le marquage initial.
3. Tant que **S** est non vide :
   1. **Dépiler** le sommet *M*.
   2. Pour chaque transition *t* admissible depuis *M*, calculer *M′* obtenu après franchissement de *t*.
   3. Si *M′* **n’est pas** dans **V**, l’ajouter et **empiler** *M′*.
   4. Ajouter l’arc **M → M′** étiqueté par *t* à **E**.

**Question 1.** Implémentez cet algorithme.

In [ ]:
def graphe(P, T, pre, post, m):
    # Génère le graphe des marquages accessibles d'un RdP borné
    # V : liste des marquages accessibles
    # E : liste des arcs (M, t, M') où t est l'étiquette de la transition

    m0 = list(m)              # on normalise le marquage initial en liste
    S = [m0]                  # pile
    V = [m0]                  # sommets déjà découverts
    E = []                    # arcs

    while S != []:
        M = S.pop()           # dépiler

        for t in franchissable(P, T, pre, post, M):
            M_prime = apply(P, T, pre, post, M, t)

            if M_prime not in V:
                V.append(M_prime)
                S.append(M_prime)

            E.append((M, t, M_prime))

    return (V, E)

# ===== TEST 1 : petit réseau borné =====
P3 = 2
T3 = 3
pre3 = [
    [1, 0, 0],
    [0, 1, 1]
]
post3 = [
    [0, 1, 0],
    [1, 0, 0]
]
m3 = [1, 0]

V3, E3 = graphe(P3, T3, pre3, post3, m3)

print("Sommets accessibles V :")
for v in V3:
    print(v)

print("Arcs E :")
for e in E3:
    print(e)

print("Nombre de sommets :", len(V3))
print("Nombre d'arcs :", len(E3))

# ===== TEST 2 : auto-boucle =====
P_test = 1
T_test = 1
pre_test = [[1]]
post_test = [[1]]
m_test = [1]

Vt, Et = graphe(P_test, T_test, pre_test, post_test, m_test)

print("\nVt =", Vt)
print("Et =", Et)

Sommets accessibles V :
[1, 0]
[0, 1]
[0, 0]
Arcs E :
([1, 0], 0, [0, 1])
([0, 1], 1, [1, 0])
([0, 1], 2, [0, 0])
Nombre de sommets : 3
Nombre d'arcs : 3

Vt = [[1]]
Et = [([1], 0, [1])]


## Exercice 3 — Model checking

**Question 1.** Écrivez une fonction qui vérifie si un RdP **borné** est **bloquant**, ainsi qu’une fonction qui, le cas échéant, renvoie un **marquage bloqué**.


In [26]:
def estbloquant(P,T,pre,post,m):
    # teste si un reseau est bloquant
        return marquagebloque(P, T, pre, post, m) is not None

def marquagebloque(P,T,pre,post,m):
    # renvoie un marquage bloqué
    V, E = graphe(P, T, pre, post, m)

    for M in V:
        if franchissable(P, T, pre, post, M) == []:
            return M
    return None



**Question 2.** Écrivez une fonction qui vérifie si un RdP **borné** est **propre** et, le cas échéant, renvoie un marquage accessible *M* à partir duquel il est **impossible de revenir** au marquage initial.


In [27]:
def successeurs(E, M):
    # renvoie la liste des couples (t, M') tels que M --t--> M'
    return [(t, M2) for (M1, t, M2) in E if M1 == M]


def predecesseurs(E, M):
    # renvoie la liste des couples (M', t) tels que M' --t--> M
    return [(M1, t) for (M1, t, M2) in E if M2 == M]


def accessibles_depuis(E, M):
    # BFS/DFS sur le graphe des marquages accessibles,
    # à partir d'un marquage M déjà présent dans le graphe
    vus = [M]
    pile = [M]

    while pile:
        courant = pile.pop()
        for (_, suivant) in successeurs(E, courant):
            if suivant not in vus:
                vus.append(suivant)
                pile.append(suivant)

    return vus

def estpropre(P,T,pre,post,m):
    # teste si un reseau est propre
    return marquage_non_propre(P, T, pre, post, m) is None

def marquage_non_propre(P, T, pre, post, m):
    # renvoie un marquage non propre
    m0 = list(m)
    V, E = graphe(P, T, pre, post, m)

    for M in V:
        atteignables = accessibles_depuis(E, M)
        if m0 not in atteignables:
            return M

    return None


**Question 3.** Écrivez une fonction qui vérifie si un réseau de Petri **borné** est **quasi-vivant** et, le cas échéant, renvoie une **transition** qui ne l’est pas.
Écrire une fonction qui teste si un réseau de Petri borné est quasi-vivant et qui renvoie une transition non-quasi-vivante le cas échéant.

In [28]:
def transition_non_qv(P, T, pre, post, m):
    # renvoie une transition non quasi-vivante
    V, E = graphe(P, T, pre, post, m)

    transitions_vues = set()
    for M in V:
        for t in franchissable(P, T, pre, post, M):
            transitions_vues.add(t)

    for t in range(T):
        if t not in transitions_vues:
            return t

    return None


def estqv(P, T, pre, post, m):
    # teste si un reseau est quasi-vivant
    return transition_non_qv(P, T, pre, post, m) is None

**Question 4.** Écrivez une fonction qui vérifie si un réseau de Petri **borné** est **vivant** et, le cas échéant, renvoie un **marquage accessible M** ainsi qu’une **transition** qui **n’est pas vivante** à partir de *M*.
Écrire une fonction qui teste si un réseau de Petri borné est vivant et qui renvoie un marquage accessible M et une transition non-vivante à partir de M le cas échéant.

In [29]:
def temoin_non_vivant(P, T, pre, post, m):
    # renvoie un témoin de non-vivacité
    V, E = graphe(P, T, pre, post, m)

    for M in V:
        atteignables = accessibles_depuis(E, M)

        for t in range(T):
            peut_redevenir_franchissable = False

            for N in atteignables:
                if t in franchissable(P, T, pre, post, N):
                    peut_redevenir_franchissable = True
                    break

            if not peut_redevenir_franchissable:
                return (M, t)

    return None


def estvivant(P, T, pre, post, m):
    # teste si un reseau est vivant
    return temoin_non_vivant(P, T, pre, post, m) is None


### Tests Exercice 3

In [30]:
# ===== Réseau A : plus grand réseau borné et cyclique =====

# Places:
# p0 : idle
# p1 : stage1
# p2 : stage2
# p3 : critical section
# p4 : finished
# p5 : resource token

P_A = 6
T_A = 7

# transitions:
# t0 : p0 -> p1
# t1 : p1 -> p2
# t2 : p2 -> p2      (self-loop / check)
# t3 : p2 + p5 -> p3
# t4 : p3 -> p4 + p5
# t5 : p4 -> p0
# t6 : p4 -> p1      (restart variant)

pre_A = [
    [1,0,0,0,0,0,0],  # p0
    [0,1,0,0,0,0,0],  # p1
    [0,0,1,1,0,0,0],  # p2
    [0,0,0,0,1,0,0],  # p3
    [0,0,0,0,0,1,1],  # p4
    [0,0,0,1,0,0,0],  # p5
]

post_A = [
    [0,0,0,0,0,1,0],  # p0
    [1,0,0,0,0,0,1],  # p1
    [0,1,1,0,0,0,0],  # p2
    [0,0,0,1,0,0,0],  # p3
    [0,0,0,0,1,0,0],  # p4
    [0,0,0,0,1,0,0],  # p5
]

m_A = [1,0,0,0,0,1]

V_A, E_A = graphe(P_A, T_A, pre_A, post_A, m_A)

print("===== RESEAU A =====")
print("Nombre de marquages accessibles :", len(V_A))
print("Marquages accessibles :")
for M in V_A:
    print(M)

print("\nBloquant ?", estbloquant(P_A, T_A, pre_A, post_A, m_A))
print("Marquage bloque :", marquagebloque(P_A, T_A, pre_A, post_A, m_A))

print("\nPropre ?", estpropre(P_A, T_A, pre_A, post_A, m_A))
print("Temoin non propre :", marquage_non_propre(P_A, T_A, pre_A, post_A, m_A))

print("\nQuasi-vivant ?", estqv(P_A, T_A, pre_A, post_A, m_A))
print("Transition non quasi-vivante :", transition_non_qv(P_A, T_A, pre_A, post_A, m_A))

print("\nVivant ?", estvivant(P_A, T_A, pre_A, post_A, m_A))
print("Temoin non vivant :", temoin_non_vivant(P_A, T_A, pre_A, post_A, m_A))

===== RESEAU A =====
Nombre de marquages accessibles : 5
Marquages accessibles :
[1, 0, 0, 0, 0, 1]
[0, 1, 0, 0, 0, 1]
[0, 0, 1, 0, 0, 1]
[0, 0, 0, 1, 0, 0]
[0, 0, 0, 0, 1, 1]

Bloquant ? False
Marquage bloque : None

Propre ? True
Temoin non propre : None

Quasi-vivant ? True
Transition non quasi-vivante : None

Vivant ? True
Temoin non vivant : None


In [ ]:
# ===== Réseau B : borné mais avec blocage et transition inutile =====

# Places:
# p0 : idle
# p1 : stage1
# p2 : stage2
# p3 : stage3
# p4 : sink / fin
# p5 : resource

P_B = 6
T_B = 7

# transitions:
# t0 : p0 -> p1
# t1 : p1 -> p2
# t2 : p2 + p5 -> p3
# t3 : p3 -> p4 + p5
# t4 : p2 -> p2      (loop local)
# t5 : p1 -> p1      (loop local)
# t6 : demande 2 jetons dans p4 -> impossible, jamais franchissable

pre_B = [
    [1,0,0,0,0,0,0],  # p0
    [0,1,0,0,0,1,0],  # p1
    [0,0,1,0,1,0,0],  # p2
    [0,0,0,1,0,0,0],  # p3
    [0,0,0,0,0,0,2],  # p4
    [0,0,1,0,0,0,0],  # p5
]

post_B = [
    [0,0,0,0,0,0,0],  # p0
    [1,0,0,0,0,1,0],  # p1
    [0,1,0,0,1,0,0],  # p2
    [0,0,1,0,0,0,0],  # p3
    [0,0,0,1,0,0,0],  # p4
    [0,0,0,1,0,0,0],  # p5
]

m_B = [1,0,0,0,0,1]

V_B, E_B = graphe(P_B, T_B, pre_B, post_B, m_B)

print("===== RESEAU B =====")
print("Nombre de marquages accessibles :", len(V_B))
print("Marquages accessibles :")
for M in V_B:
    print(M)

print("\nBloquant ?", estbloquant(P_B, T_B, pre_B, post_B, m_B))
print("Marquage bloque :", marquagebloque(P_B, T_B, pre_B, post_B, m_B))

print("\nPropre ?", estpropre(P_B, T_B, pre_B, post_B, m_B))
print("Temoin non propre :", marquage_non_propre(P_B, T_B, pre_B, post_B, m_B))

print("\nQuasi-vivant ?", estqv(P_B, T_B, pre_B, post_B, m_B))
print("Transition non quasi-vivante :", transition_non_qv(P_B, T_B, pre_B, post_B, m_B))

print("\nVivant ?", estvivant(P_B, T_B, pre_B, post_B, m_B))
print("Temoin non vivant :", temoin_non_vivant(P_B, T_B, pre_B, post_B, m_B))

## Exercice 4 — Traverser la rivière
Un groupe de 4 personnes (**A, B, C, D**) doivent traverser un pont mal éclairé pour franchir une rivière.  
Le pont ne supporte le poids que de **deux personnes**. Ils ont à disposition **une seule lampe de poche** (qui doit être allumée pour chaque traversée).  
Pour traverser le pont, A met **10 min**, B met **5 min**, C met **2 min** et D met **1 min**.

**Objectif.** Déterminer le **temps minimal** nécessaire pour que tout le groupe traverse le pont.

**Question 1.** Représentez ce problème sous la forme d’un **RdP** et en déduire le **temps minimal** de la traversée.


In [23]:
import heapq

# Personnes : A=0 (10min), B=1 (5min), C=2 (2min), D=3 (1min)
durees = [10, 5, 2, 1]
noms = ['A', 'B', 'C', 'D']

# Places :
# 0-3 : A,B,C,D a gauche
# 4-7 : A,B,C,D a droite
# 8   : torche a gauche
# 9   : torche a droite
P4 = 10

# Transitions :
# - 6 paires allant vers la droite (torche gauche -> droite)
# - 4 singles allant vers la droite
# - 4 singles revenant vers la gauche (torche droite -> gauche)
# - 6 paires revenant vers la gauche
pairs   = [(0,1),(0,2),(0,3),(1,2),(1,3),(2,3)]
singles = [0,1,2,3]

transitions_list = []
durees_t = []

for (i,j) in pairs:     # paires vers la droite
    transitions_list.append(('right', i, j)); durees_t.append(max(durees[i], durees[j]))
for i in singles:        # singles vers la droite
    transitions_list.append(('right', i, None)); durees_t.append(durees[i])
for i in singles:        # singles vers la gauche
    transitions_list.append(('left', i, None)); durees_t.append(durees[i])
for (i,j) in pairs:     # paires vers la gauche
    transitions_list.append(('left', i, j)); durees_t.append(max(durees[i], durees[j]))

T4 = len(transitions_list)

# Construction des matrices pre et post
pre4  = [[0]*T4 for _ in range(P4)]
post4 = [[0]*T4 for _ in range(P4)]

for t_idx, (direction, i, j) in enumerate(transitions_list):
    if direction == 'right':
        # personnes quittent la gauche, torche quitte la gauche
        pre4[i][t_idx]    = 1;  pre4[8][t_idx]    = 1
        post4[i+4][t_idx] = 1;  post4[9][t_idx]   = 1
        if j is not None:
            pre4[j][t_idx]    = 1
            post4[j+4][t_idx] = 1
    else:  # 'left'
        # personnes quittent la droite, torche quitte la droite
        pre4[i+4][t_idx]  = 1;  pre4[9][t_idx]    = 1
        post4[i][t_idx]   = 1;  post4[8][t_idx]   = 1
        if j is not None:
            pre4[j+4][t_idx] = 1
            post4[j][t_idx]  = 1

# Marquage initial : tout le monde a gauche, torche a gauche
m4      = [1,1,1,1,0,0,0,0,1,0]
# Marquage but : tout le monde a droite, torche a droite
m4_goal = [0,0,0,0,1,1,1,1,0,1]

# Dijkstra pour trouver le chemin de temps minimal
def temps_min(P, T, pre, post, m_init, m_goal, durees_t):
    m_init_t = tuple(m_init)
    m_goal_t = tuple(m_goal)

    dist = {m_init_t: 0}
    prev = {m_init_t: (None, None)}   # m_precedent, transition utilisee
    heap = [(0, m_init_t)]

    while heap:
        d, m_t = heapq.heappop(heap)

        if m_t == m_goal_t:
            # Reconstruction du chemin
            path = []
            curr = m_t
            while prev[curr][0] is not None:
                path.append(prev[curr])
                curr = prev[curr][0]
            path.reverse()
            return d, path

        if d > dist[m_t]:
            continue

        m = list(m_t)
        for t in franchissable(P, T, pre, post, m):
            m_prime_t = tuple(apply(P, T, pre, post, m, t))
            new_d = d + durees_t[t]
            if m_prime_t not in dist or new_d < dist[m_prime_t]:
                dist[m_prime_t] = new_d
                prev[m_prime_t] = (m_t, t)
                heapq.heappush(heap, (new_d, m_prime_t))

    return None, None  # but non accessible

temps, chemin = temps_min(P4, T4, pre4, post4, m4, m4_goal, durees_t)

print(f"temps minimal : {temps} minutes\n")

for (m_from, t) in chemin:
    direction, i, j = transitions_list[t]
    fleche = '->' if direction == 'right' else '<-'
    if j is not None:
        print(f"  {noms[i]} + {noms[j]}  {fleche}  ({durees_t[t]} min)")
    else:
        print(f"  {noms[i]}  {fleche}  ({durees_t[t]} min)")

temps minimal : 17 minutes

  C + D  ->  (2 min)
  D  <-  (1 min)
  A + B  ->  (10 min)
  C  <-  (2 min)
  C + D  ->  (2 min)


## Exercice 5 — Protocole
Vous considérez un **protocole** de connexion/déconnexion entre un **client** et un **serveur**.

### Connexion
1. Le **client** initie la connexion en envoyant une **demande de connexion (DC)**, puis **attend**.
2. À la réception de **DC**, le **serveur** envoie **CC (confirmation de connexion)** puis passe dans l’**état connecté**.
3. Le **client** passe dans l’**état connecté** à la réception de **CC**.

### Déconnexion
1. Le **client (ou le serveur)** envoie une **demande de déconnexion** **DD1 (ou DD2)** puis **attend**.
2. À la réception de **DD1 (ou DD2)**, le **serveur (ou le client)** envoie une **confirmation de déconnexion** **CD1 (ou CD2)**, puis passe dans l’**état déconnecté**.
3. À la réception de **CD1 (ou CD2)**, le **client (ou le serveur)** passe dans l’**état déconnecté**.

**Question 1.** Représentez ce protocole à l’aide d’un **RdP**. Est-il **borné** ? **bloquant** ? Si oui, **justifiez**.

#### Modélisation

On a 13 places : 
- 0 : client déconnecté
- 1 : client envoyé DC, attente CC
- 2 : client connecté
- 3 : client envoyé DD1, attente CD1
- 4 : serveur déconnecté
- 5 : serveur connecté
- 6 : serveur envoyé DD2, attente CD2
- 7 : DC
- 8 : CC
- 9 : DD1
- 10 : CD1
- 11 : DD2
- 12 : CD2

On a 9 transitions :
- 0 : client envoie DC
- 1 : serveur reçoit DC, envoie CC
- 2 : client reçoit CC
- 3 : client envoie DD1
- 4 : serveur reçoit DD1, envoie CD1
- 5 : client reçoit CD1
- 6 : serveur envoie DD2
- 7 : client reçoit DD2, envoie CD2
- 8 : serveur reçoit CD2

In [31]:
P = 13
T = 9

pre  = [[0]*T for _ in range(P)]
post = [[0]*T for _ in range(P)]

# Transition 0
pre[0][0]  = 1
post[1][0] = 1
post[7][0]  = 1

# Transition 1
pre[4][1]  = 1
pre[7][1]   = 1
post[5][1] = 1
post[8][1]  = 1

# Transition 2
pre[1][2]  = 1
pre[8][2]   = 1
post[2][2] = 1

# Transition 3
pre[2][3]  = 1
post[3][3] = 1
post[9][3]  = 1

# Transition 4
pre[5][4]  = 1
pre[9][4]   = 1
post[4][4] = 1
post[10][4] = 1

# Transition 5
pre[3][5]  = 1
pre[10][5]  = 1
post[0][5] = 1

# Transition 6
pre[5][6]  = 1
post[6][6] = 1
post[11][6] = 1

# Transition 7
pre[2][7]  = 1
pre[11][7]  = 1
post[0][7] = 1
post[12][7] = 1

# Transition 8
pre[6][8]  = 1
pre[12][8]  = 1
post[4][8] = 1

# Marquage initial : client + serveur déconnectés (places 0 et 5), canaux vides
m = [1,0,0,0,1,0,0,0,0,0,0,0,0]

print('Borné    :', estborne(P, T, pre, post, m))
print('Bloquant :', estbloquant(P, T, pre, post, m))
marquage = marquagebloque(P, T, pre, post, m)
print('Marquage bloqué:', marquage)
print('Indice des places du marquage bloqué:', [i for i, val in enumerate(marquage) if val > 0])


Borné    : True
Bloquant : True
Marquage bloqué: [0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0]
Indice des places du marquage bloqué: [3, 6, 9, 11]


#### Interprétation

Les places 3, 6, 9 et 11 sont bloquées.
Il s'agit de :
- 3 : client envoyé DD1, attente CD1
- 6 : serveur envoyé DD2, attente CD2
- 9 : DD1
- 11 : DD2

Ici on dirait qu'il y a eu un interblocage : chaque partie attend une confirmation de déconnexion que l'autre partie doit envoyer.